In [6]:
import requests
import json

full_text_search_term = 'where do cabybara live?'

url = 'https://en.wikipedia.org/w/api.php?'

params = {
    'action': 'query',
    'format': 'json', 
    'list': 'search',
    'srsearch': full_text_search_term
    
}
headers = {
    'User-Agent': 'Student Project'
}

response = requests.get(url, params=params, headers=headers)
response.raise_for_status()

data = response.json()

In [16]:
for i in data['query']['search']:
    print(i['title'])

Capybara


In [17]:
url = 'https://en.wikipedia.org/w/index.php?'

params = {
    'title':'Capybara',
    'action':'raw',
}
headers = {
        'User-Agent': 'Student Project'
    }

response = requests.get(url, params=params, headers=headers)
response.raise_for_status()


In [20]:
capybara_doc = response.text

In [22]:
generator_instructions = """ 
You are given a Wikipedia article. Your task is to imagine what a person might ask when looking something up on Wikipedia.

Generate realistic questions that humans might ask when looking something up in an encyclopedia. Users are diverse with varied educational level and adjust your questions accordingly. 

Guidelines:
- Questions should reflect varied levels of expertise, from amateur to expert.
- Make questions varied and spontaneous, not repetitive or over-polished.

Distribution of rules:
- 60% of queries should target amateur-level users, looking to get a superficial understanding.
- 20% of queries should target enthusiast who are looking for more depth regarding the topic.
- 20% of queries should target experts who are familiar with the topic who are looking for details and anomalies.

For each generated query, include:
- question: reflecting varied knowledge levels
- summary_answer: a short 1-2 sentence summary of how the article addresses it
- difficulty: one of ['amateur', 'enthusiast', 'expert']
"""

user_prompt = f"Generate 20 questions for this document: {capybara_doc}"
user_prompt

'Generate 20 questions for this document: {{Short description|Largest species of rodents}}\n{{Other uses}}\n{{Good article}}\n{{pp|small=yes}}\n{{Use dmy dates|date=July 2022}}\n{{Speciesbox\n| status            = LC\n| status_system     = IUCN3.1\n| status_ref        = <ref name="iucn status 19 November 2021">{{cite iucn |author=Reid, F. |date=2016 |title=\'\'Hydrochoerus hydrochaeris\'\' |volume=2016 |article-number=e.T10300A22190005 |doi=10.2305/IUCN.UK.2016-2.RLTS.T10300A22190005.en |access-date=19 November 2021}}</ref>\n| image             = Hydrochoeris hydrochaeris in Brazil in Petrópolis, Rio de Janeiro, Brazil 09.jpg\n| image_caption     = In [[Petrópolis]], Brazil\n| genus             = Hydrochoerus\n| species           = hydrochaeris\n| authority         = ([[Carl Linnaeus|Linnaeus]], [[12th edition of Systema Naturae|1766]])\n| range_map         = Capybara range.svg\n| range_map_caption = Native range\n| synonyms          = \'\'Sus hydrochaeris\'\' {{small|Linnaeus,&nbsp;17

In [23]:
from pydantic import BaseModel, Field
from typing import List, Literal

class Question(BaseModel):
    """
    Represents a realistic search-engine-style query a user might type before finding the article.
    Each question captures the likely search phrase, a short summary answer,
    the user's assumed knowledge level.
    """
    question: str = Field(
        ...,
        description="A natural, short search query — not a full-sentence question — phrased like something typed into Wikipedia."
    )
    summary_answer: str = Field(
        ...,
        description="A concise 1–2 sentence summary of how the article addresses the query."
    )
    difficulty: Literal["amateur", "enthusiast", "expert"] = Field(
        ...,
        description="The assumed knowledge level of the user making the query."
    )



class GeneratedQuestions(BaseModel):
    """
    A structured collection of human-like search queries derived from a given Wikipedia article.
    Includes a brief description of the article topic and a list of generated queries.
    Difficulty distribution: 60% amateur, 30% enthusiast, 10% expert.
    """
    description: str = Field(
        ...,
        description="A summary of the article or topic these search-style questions were generated for."
    )
    questions: List[Question] = Field(
        ...,
        description="A list of realistic search queries with short summaries, difficulty levels, and user intent."
    )

In [24]:
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

def map_progress(pool, seq, f):
    """Map function f over seq using the provided executor pool while
    displaying a tqdm progress bar. Returns a list of results in submission order.
    """
    results = []
    
    with tqdm(total=len(seq)) as progress:
        futures = []
    
        for el in seq:
            future = pool.submit(f, el)
            future.add_done_callback(lambda p: progress.update())
            futures.append(future)

        for future in futures:
            result = future.result()
            results.append(result)
        
        return results

from openai import OpenAI
openai_client = OpenAI()

def llm_structured(instructions, user_prompt, output_format, model="gpt-4o-mini"):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=output_format
    )

    return (response.output_parsed, response.usage)

response, usage = llm_structured(
        instructions=generator_instructions,
        user_prompt=user_prompt,
        output_format=GeneratedQuestions)

In [ ]:
final_questions = []
for q in response.questions:
    final_question = q.model_dump()
    final_questions.append(final_question)
final_questions

In [32]:
import pandas as pd

df_questions = pd.DataFrame(final_questions)
df_questions

,question,summary_answer,difficulty
0,What is a capybara?,"The capybara, or Hydrochoerus hydrochaeris, is...",amateur
1,Where do capybaras live?,Capybaras inhabit savannas and dense forests i...,amateur
2,How big do capybaras get?,Adult capybaras can reach lengths of up to 134...,amateur
3,What do capybaras eat?,Capybaras are herbivores that primarily graze ...,amateur
4,Why are capybaras considered social animals?,Capybaras are highly social creatures that usu...,amateur
5,What is the capybara's natural habitat?,Capybaras thrive in regions with dense vegetat...,amateur
6,What threats do capybaras face?,Capybaras face threats from hunting for meat a...,amateur
7,How do capybaras communicate?,Capybaras use various vocalizations including ...,enthusiast
8,What adaptations do capybaras have for swimming?,Capybaras are excellent swimmers with webbed f...,enthusiast
9,What role do capybaras play in their ecosystem?,Capybaras impact their environment by grazing ...,enthusiast


In [34]:
df_questions.to_csv('ground_truth_cabybara.csv', index=False)